# 第4课：函数与模块——写出可复用的代码

> **学习目标**：理解函数的本质，掌握参数传递技巧，学会使用模块组织代码

---

## 为什么需要函数？——DRY 原则

**DRY = Don't Repeat Yourself（不要重复自己）**，这是软件开发的基石之一。

### 没有函数的世界

想象你开了一家奶茶店。每次有客人点单，你都要：
1. 拿出杯子 → 2. 加入茶底 → 3. 加奶/糖 → 4. 封口 → 5. 收钱

如果每次点单你都重新**写一遍这5步的完整流程**，你的"操作手册"会越来越厚，而且万一要改配方（比如换杯子），你得翻遍整本手册去改每一处。

**函数就是"把这个流程提取出来，给它起个名字，以后直接叫这个名字就行"**。

### 重复代码的三大危害

| 问题 | 表现 |
|------|------|
| **修改困难** | 改一处需求要动 N 个地方，漏掉一处就是 Bug |
| **可读性差** | 细节淹没逻辑，看代码等于看流水账 |
| **复用为零** | 换个项目就要重写，无法积累代码资产 |

### 调用函数时，Python 内部发生了什么？

当你写下 `evaluate("张三", 85)` 时，Python 解释器内部完成了一系列精密操作。理解这个过程，能帮你真正理解函数的本质。

**第一步：压栈（Push）**。Python 将当前执行位置（返回地址）和当前局部变量的引用压入"调用栈"（Call Stack）。调用栈是程序内存中的一块区域，用"后进先出"（LIFO）的方式管理函数调用。每调用一个函数，栈就长高一层；每返回一个函数，栈就缩短一层。你可以把调用栈想象成一个"层层叠叠的托盘"——最后放上去的盘子最先被拿走。

**第二步：创建帧对象（Frame Object）**。Python 为这次调用创建一个帧对象，它包含了函数的局部变量、参数值、当前执行到的字节码位置等信息。每个活跃的函数调用都有自己独立的帧对象，互不干扰。你可以在一个函数中调用另一个函数，这个函数再调用下一个——每一层都有自己的帧，它们按调用顺序叠在栈上。

**第三步：参数绑定（Argument Binding）**。Python 将传入的参数值按照函数定义的参数规则（位置、默认值、关键字、*args、**kwargs）绑定到参数变量上。这个绑定过程是 Python 参数系统最灵活、也设计最精妙的部分。

**第四步：执行字节码**。Python 将函数体编译成的字节码在帧对象中逐条执行。当遇到 `return` 语句时，返回值被传回调用方，同时当前帧对象被销毁，调用栈"弹出"一层，恢复调用方的执行。

这就是为什么函数调用是有"开销"的一一每次调用都要完成栈操作、帧创建、参数绑定等一系列步骤。但也正是这套机制，让函数成为了代码复用的理想单位。

下面看看重复代码 vs 函数的具体对比。

In [ ]:
# ================================
# 没有函数：WET（Write Everything Twice）
# ================================
# WET 原则：Write Everything Twice —— 重复编写相似逻辑的反模式
# 本段演示了如果不使用函数，相同的判断逻辑需要在多处重复编写

name1, score1 = "张三", 85
name2, score2 = "李四", 92
name3, score3 = "王五", 78
# 上面三行：分别定义了三组学生姓名和分数的变量
# 每多一个学生，就需要多定义一组变量，代码迅速膨胀

# 三段几乎一样的逻辑
# 下面这段 if/elif/else 结构完整地重复了三次
# 每一段都根据分数判断等级并打印输出
# 注意：这三段代码之间唯一的区别只是变量名 name1/score1、name2/score2、name3/score3 不同
# 其余逻辑完全一致 —— 这就是"重复代码"的典型症状
if score1 >= 90:
    print(f"{name1}: 优秀")
elif score1 >= 60:
    print(f"{name1}: 及格")
else:
    print(f"{name1}: 不及格")

if score2 >= 90:
    print(f"{name2}: 优秀")
elif score2 >= 60:
    print(f"{name2}: 及格")
else:
    print(f"{name2}: 不及格")

if score3 >= 90:
    print(f"{name3}: 优秀")
elif score3 >= 60:
    print(f"{name3}: 及格")
else:
    print(f"{name3}: 不及格")

# 三处完全相同的判断逻辑！
# 如果要改评级标准：改三处，漏一处就出 Bug
# 实际问题：如果将及格线从 60 改为 65，需要手动修改三个地方的 elif 条件
# 如果漏改了一处，就会导致部分学生使用旧标准，产生不一致的结果

In [ ]:
# ================================
# 有函数：DRY
# ================================
# DRY 原则：Don't Repeat Yourself —— 将重复逻辑提取为函数，一次定义多次调用

def evaluate(name, score):
    """根据分数评价学生"""
    # def 关键字：定义一个新函数
    # evaluate：函数名，采用 snake_case 命名风格（动词或动名词）
    # name, score：两个位置参数，调用时按顺序传入
    # """根据分数评价学生"""：文档字符串（docstring），说明函数功能
    # 此参数接收一个学生的姓名和分数，返回对应的评语字符串
    if score >= 90:
        return f"{name}: 优秀"
    elif score >= 60:
        return f"{name}: 及格"
    else:
        return f"{name}: 不及格"
    # return 语句：将结果返回给调用方，并立即结束函数执行
    # 这里返回的是一个 f-string 格式的字符串

# 一次定义，反复使用
# 下面三次调用 evaluate 函数，每次传入不同的参数
# 调用函数时，Python 的执行流程：
#   1. 暂停当前代码的执行
#   2. 将参数值按位置赋给函数参数（name="张三", score=85）
#   3. 跳转到函数定义处执行函数体
#   4. 遇到 return 后带着返回值跳回原调用位置
#   5. 将返回值传递给 print() 输出
print(evaluate("张三", 85))
print(evaluate("李四", 92))
print(evaluate("王五", 78))

# 要改评级标准？只改 evaluate 内部即可！
# 要增加学生？加一行调用即可！
# 相比 WET 版本，这里实现了"一处定义、多处复用"
# 后续如果需要修改评级逻辑，只需要修改 evaluate 函数内部的判断条件
# 所有调用点自动生效，不会出现漏改的情况

---

## 函数定义与调用

### 定义语法

```python
def 函数名(参数1, 参数2, ...):
    """文档字符串（可选，但推荐）"""
    # 函数体（缩进）
    return 返回值
```

- **`def`**：关键词，告诉 Python "我在定义一个函数"
- **函数名**：遵循 `snake_case`，应该是动词或动名词
- **参数**：函数的"原料"
- **函数体**：缩进的代码块——函数的"加工步骤"
- **`return`**：函数的"成品"。没有 return 的函数返回 `None`

### 调用语法

```python
函数名(参数值1, 参数值2)
```

### 调用栈：函数调用的"记账本"

调用栈（Call Stack）是理解函数嵌套调用的关键概念。当你写：

```python
def a():
    b()

def b():
    c()

def c():
    pass

a()
```

调用 `a()` 时，Python 把 a 的调用信息压入栈；a 调用了 b，b 的信息压在 a 上面；b 又调用了 c，c 压在 b 上面。当 c 返回时，c 的帧从栈顶弹出，恢复 b 的执行；b 返回后，b 的帧弹出，恢复 a 的执行；a 返回后，栈恢复为空。

这就是"后进先出"（LIFO）——最后被调用的函数最先返回。如果你在程序崩溃时看到长长的"Traceback"，那其实就是 Python 在打印当前调用栈的内容，每一帧对应一个未返回的函数调用。

### 参数绑定：实参是怎样变成形参的？

当你调用 `evaluate("张三", 85)` 时，Python 做了参数绑定（Argument Binding）：
1. 检查传入的实参个数是否与形参个数匹配
2. 按位置将 "张三" 绑定到 name，85 绑定到 score
3. 如果有关键字参数，按参数名匹配
4. 如果有默认参数，未传入的用默认值填充
5. 如果有 `*args`，多余位置参数打包成元组
6. 如果有 `**kwargs`，多余关键字参数打包成字典

这一整套规则保证了参数传递的灵活性和安全性。

### 类比：快递柜取件

- **函数定义** = 在快递柜上贴标签（"1号柜"）
- **参数** = 取件码（告诉快递柜取哪个件）
- **调用** = 输入取件码
- **返回值** = 柜门打开，拿到包裹

In [ ]:
# ================================
# 定义与调用基础
# ================================

def greet(name):
    """向指定的人打招呼"""
    # def greet(name): —— 定义名为 greet 的函数，接收一个位置参数 name
    # name：参数变量，在函数体内代表调用时传入的值
    # """向指定的人打招呼"""：文档字符串，说明函数功能
    # 返回值：一个包含问候语的字符串
    return f"你好，{name}！"
    # f-string 格式化字符串：将变量 name 的值嵌入到字符串中
    # return 将结果字符串返回给调用方

# 调用函数——使用返回值
# 调用 greet("小明")，Python 将字符串 "小明" 赋值给参数 name
# 执行函数体，return 返回 "你好，小明！"
# 返回值被赋给变量 msg
msg = greet("小明")
print(msg)

# 直接打印返回值
# 也可以不经过中间变量，直接将函数调用作为 print() 的参数
print(greet("小红"))
print(greet("小刚"))

# 函数可以调用任意多次
# 在 for 循环中反复调用同一个函数，传入不同的参数值
# 这是函数复用的典型场景：一次定义，循环调用
for student in ["张三", "李四", "王五"]:
    print(greet(student))

### return：函数的"回报"

`return` 是函数和外部沟通的桥梁。理解它的三个关键点：

1. **`return` 立即结束函数** — 后面的代码不会执行（类似循环的 `break`）
2. **没有 return = 返回 `None`** — Python 偷偷在函数末尾加了 `return None`
3. **可以返回多个值** — 本质是返回一个元组，然后解包

### `None` 是什么？

`None` 是 Python 的"空值"，表示"什么都没有"。它不等于 `False`、`0` 或空字符串——它是一个**独立类型** `NoneType` 的**唯一值**。Python 解释器在启动时就创建好了这个唯一的 None 对象，之后所有用到 None 的地方，引用的都是同一个对象。你可以用 `None is None` 来验证——`is` 比较的是对象身份（内存地址），结果为 True，说明所有 None 确实是同一个东西。

> 很多 Bug 的根源：你以为函数返回了结果，实际上它返回了 `None`。然后你对 `None` 调用了方法（比如 `.append()`），程序就炸了。

### 为什么是 None，不是 null？

如果你用过 Java 或 JavaScript，你可能见过 `null`。Python 选择 `None` 这个名字，体现了它的设计哲学——"没有"不是"空"，而是一个有明确含义的独立值。在 Python 中：
- 函数没有 `return` → 返回 None（表示"没有返回任何有意义的值"）
- 字典取值没有 key → 返回 None（通过 `.get()` 方法）
- 变量还没有值 → 可以显式赋值为 None（作为"尚未赋值"的标记）

一个实用建议：永远用 `is None` 而不是 `== None` 来判断空值。因为 `is` 比较的是对象身份（是否是同一个 None 对象），而 `==` 比较的是值相等——理论上你可以重写 `__eq__` 方法让任何对象 `== None` 返回 True，但 `is None` 是绝对安全的。

In [ ]:
# ================================
# return 深入理解
# ================================
# 本段演示 return 的三个关键特性

# 1. return 立即结束函数
def check(score):
    # check 函数接收一个分数参数，返回评语字符串
    if score < 0:
        return "无效分数"    # 到这里就返回了
    # 如果 score < 0 为 True，执行到上面的 return 后函数立即结束
    # 下面这行 return 语句在此情况下不会被执行
    # 下面这行只有 score >= 0 才执行
    return f"分数: {score}"
    # 每个 return 都是一个独立的退出点
    # 函数可以有多个 return，但只会执行其中一个

print(check(-5))   # "无效分数"
print(check(80))   # "分数: 80"

# 2. 没有 return → 返回 None
def no_return(x):
    # 这个函数体只有一个表达式 x * 2
    # 注意：此处虽然计算了 x * 2，但没有 return 语句返回结果
    # 计算结果被丢弃了！这是一个常见的错误
    x * 2           # 计算了，但没返回

result = no_return(5)
# no_return(5) 执行了 x * 2（结果是 10），但没有 return
# Python 隐式在函数末尾添加了 return None
# 所以 result 被赋值为 None
print(f"没有 return 的结果: {result}")    # None
# None 是 NoneType 类型的唯一值，表示"没有值"
# 在布尔上下文中 None 被视为 False，但 None 不等于 False
print(f"类型是: {type(result)}")          # <class 'NoneType'>

# 3. 返回多个值——自动打包成元组
def stats(numbers):
    # stats 函数接收一个数字列表，返回三个统计指标
    # Python 允许一次 return 多个值，用逗号分隔
    # 实际上，return 多个值时 Python 自动将它们打包成一个元组 (min, max, avg)
    return min(numbers), max(numbers), sum(numbers) / len(numbers)

# 多重赋值（解包）：将返回的元组拆开赋给三个变量
# 注意：变量数量必须与返回值的数量一致，否则会引发 ValueError
low, high, avg = stats([3, 1, 4, 1, 5, 9])
# 等价于：先接收元组 result = (1, 9, 3.833...)，再解包 low, high, avg = result
print(f"最小: {low}, 最大: {high}, 平均: {avg:.1f}")
# :.1f 是格式化说明符，保留一位小数

---

## 参数系统：函数与外部通信的桥梁

Python 的参数系统极其灵活，理解它是写出好函数的关键。

| 类型 | 语法 | 类比 | 场景 |
|------|------|------|------|
| 位置参数 | `def f(a, b)` | 按座位坐 | 最基本，一一对应 |
| 默认参数 | `def f(a, b=10)` | 餐厅默认套餐 | 大部分情况有默认值 |
| 关键字参数 | `f(a=1, b=2)` | 喊名字点菜 | 明确参数含义 |
| `*args` | `def f(*args)` | 自助餐随便夹 | 参数个数不确定 |
| `**kwargs` | `def f(**kwargs)` | 开放点单 | 需要任意选项 |

### 位置参数：最简单的"对号入座"

调用函数时传入的值按**位置顺序**赋给参数。

### 为什么 Python 要设计这么多参数类型？

这不是 Python 没事找事——每一种参数类型都解决了特定的问题：

- **位置参数**是所有语言最基本的形式，简单、直接、一目了然。当你调用 `pow(2, 3)` 时，看到两个数字按顺序传进去，任何人都能理解。

- **默认参数**解决的是"大部分情况下用一个值，偶尔需要改"的问题。比如 `print()` 的 `sep=' '` 参数——绝大多数时候用空格分隔就够了，但你可以改成 `sep=','`。没有默认参数，你就得每次调用都写 `print("a", "b", sep=' ')`。默认参数让函数调用变得简洁，又不失灵活性。

- **关键字参数**让调用意图变得自文档化。`create_user(name="小明", age=25)` 比 `create_user("小明", 25)` 清晰得多——你不必去查函数定义就知道每个参数的含义。而且顺序不重要了，减少记忆负担。当你看到一个函数有 4 个以上的参数时，写关键字参数调用是基本礼貌。

- **`*args` 和 `**kwargs`** 解决的是"不确定参数个数"的问题。`print()` 可以接受 1 个参数也可以接受 10 个，这就是 `*args` 的功劳。框架代码中大量使用 `**kwargs` 来透传配置选项，这是 Python 代码灵活性的重要来源。

Python 的哲学是**显式优于隐式**（Explicit is better than implicit）。参数系统让函数调用变得明确——你传入什么、怎么传，都有清晰的规定。每一种参数类型都是为特定场景设计的工具，理解它们的"为什么"比死记语法更重要。

### 位置参数：最简单的"对号入座"

调用函数时传入的值按**位置顺序**赋给参数。

In [ ]:
# ================================
# 位置 / 默认 / 关键字参数
# ================================
# 本段演示 Python 三种基本的参数传递方式

# 位置参数：按顺序一一对应
def power(base, exp):
    # def power(base, exp): —— 两个位置参数
    # 调用时必须按 (底数, 指数) 的顺序传入
    # 返回：base 的 exp 次幂（base ** exp）
    return base ** exp

print(power(2, 3))   # 8 — 2→base, 3→exp
# 位置参数的关键：调用时参数的"位置"决定了它赋给哪个参数变量
# power(3, 2) 则 base=3, exp=2，结果是 9，含义完全不同

# 默认参数：不给就用预设值
def greet(name, greeting="你好"):
    # greeting="你好"：默认参数，调用时如果未提供该参数，则使用默认值 "你好"
    # 默认参数必须位于位置参数之后（否则会引发 SyntaxError）
    # 执行流程：print() 先计算 f-string 再输出
    print(f"{greeting}，{name}！")

greet("小明")               # 使用默认 greeting
# 只传了一个参数 "小明"，对应的 greeting 使用默认值 "你好"
greet("Tom", "Hello")      # 覆盖默认值
# 传了两个参数，greeting 被覆盖为 "Hello"

# 关键字参数：指名道姓，顺序不重要
def profile(name, age, city):
    # 三个位置参数，但可以通过关键字方式调用
    print(f"{name}, {age}岁, 来自{city}")

profile(city="上海", name="小红", age=23)
# 关键字参数通过参数名匹配，因此顺序可以任意
# 这里 city 排第一个，但 Python 按参数名赋值，与位置无关
# 关键字参数让调用意图极其清晰！

# ⚠️ 混用时：位置参数必须在关键字参数之前
profile("小刚", city="广州", age=28)    # OK
# "小刚" 是位置参数 → name
# 位置参数传完后，剩余参数 city 和 age 通过关键字方式传递
# 一旦开始使用关键字参数，后面不能再跟位置参数
# profile(city="广州", "小刚", 28)       # 语法错误！
# 上面的错误写法：关键字参数之后出现了位置参数，Python 无法解析
# 规则总结：位置参数 → 关键字参数（不可逆转）

### `*args`：接收任意数量的位置参数

`*args` 把多余的位置参数打包成一个**元组**。

```python
def sum_all(*args):
    # args = (1, 2, 3, 4, 5)
    ...
```

**`*` 是关键**，`args` 只是习惯命名。你也可以 `*numbers`。

### 为什么是元组不是列表？

Python 选择用元组来打包 `*args`，这是一个经过深思熟虑的设计决策。元组是不可变的，这意味着你不能在函数内部意外地修改 `args`。这也传递了一个信号：这些参数代表了调用时传入的原始值，不应该被篡改。这体现了 Python 对函数纯度的追求——函数应该清晰地分开"输入"（参数）和"处理逻辑"。

### 解包操作符：`*` 和 `**` 的另一面

你可能注意到了，`*` 和 `**` 不仅在函数定义时使用，在函数调用时也可以使用：

```python
def add(a, b, c):
    return a + b + c

nums = [1, 2, 3]
print(add(*nums))  # 把列表解包成三个位置参数 → add(1, 2, 3) = 6

config = {"a": 1, "b": 2, "c": 3}
print(add(**config))  # 把字典解包成关键字参数 → add(a=1, b=2, c=3) = 6
```

`*` 在调用时将序列（列表、元组等）解包为独立的位置参数；`**` 将字典解包为独立的关键字参数。这两个方向的操作（打包和解包）让 Python 的参数传递变得极其灵活。

### 实际用途

1. **你不知道参数的个数** — 比如 `print()` 可以接收任意数量的参数
2. **参数转发** — 在装饰器中最常见，原封不动传给另一个函数
3. **数学运算** — 求和、求积等

### `**kwargs`：接收任意数量的关键字参数

`**kwargs` 把多余的关键字参数打包成一个**字典**。

```python
def create_profile(**kwargs):
    # kwargs = {"name": "小明", "age": 25, "city": "北京"}
    ...
```

### 为什么用字典？

同样，选择字典而不是列表来存储 `**kwargs` 是合理的——关键字参数的键值对特性天然适合字典。同时，字典的 `.get()` 方法让安全地访问可能不存在的键变得简单。

### 实际用途

1. **灵活配置** — 函数需要处理不确定的配置选项
2. **参数透传** — 框架中极其常见（Flask、Django）
3. **包装/扩展** — 在原有函数基础上加功能，不改原函数签名

In [ ]:
# ================================
# *args 和 **kwargs
# ================================
# *args 和 **kwargs 是 Python 处理可变数量参数的机制
# * 和 ** 是真正的语法关键字，args 和 kwargs 只是约定俗成的命名
# 你也可以写成 *numbers 或 **options，星号才是关键

# *args：任意数量位置参数
def sum_all(*args):
    """求所有参数的和"""
    # *args 将所有传入的额外位置参数打包成一个元组（tuple）
    # 例如 sum_all(1, 2, 3, 4) → args = (1, 2, 3, 4)
    # 注意：args 是一个元组，不是列表 → 不可修改
    # 如果没有传入任何参数，args 是一个空元组 ()
    total = 0
    for n in args:
        total += n
    return total
    # 这里返回的是所有参数累加的和

print(sum_all(1, 2))            # 3
# 调用 sum_all(1, 2) → args = (1, 2) → 返回 1+2 = 3
print(sum_all(1, 2, 3, 4, 5))  # 15
# 调用 sum_all(1,2,3,4,5) → args = (1,2,3,4,5) → 返回 15

# **kwargs：任意数量关键字参数
def show_config(**kwargs):
    """显示配置项"""
    # **kwargs 将所有传入的关键字参数打包成一个字典（dict）
    # 例如 show_config(host="localhost", port=8080) → kwargs = {"host": "localhost", "port": 8080}
    # 如果没有传入任何关键字参数，kwargs 是一个空字典 {}
    print("配置信息：")
    for key, value in kwargs.items():
        print(f"  {key} -> {value}")
    # .items() 返回字典的键值对视图，适用于遍历

show_config(host="localhost", port=8080, debug=True)
# 三个关键字参数被打包为 {"host": "localhost", "port": 8080, "debug": True}
# 遍历时按插入顺序（Python 3.7+ 字典有序）输出

# 组合：万能日志函数
def log(level, msg, *args, **kwargs):
    """灵活的日志函数"""
    # 参数顺序：普通参数 → *args → **kwargs（固定顺序）
    # level：位置参数，必须提供（日志级别）
    # msg：位置参数，必须提供（日志消息主体）
    # *args：捕获所有额外的位置参数
    # **kwargs：捕获所有额外的关键字参数
    parts = [f"[{level}]", msg]
    parts.extend(str(a) for a in args)
    # 将 *args 中的每个参数转为字符串并追加到 parts 列表
    # 生成器表达式 (str(a) for a in args) 延迟计算，逐个转换
    print(" ".join(parts))
    # " ".join() 用空格连接列表中的所有元素
    if kwargs:
        print("  附加:", kwargs)
    # 如果有关键字参数，打印出来
    # if kwargs：空字典在布尔上下文中为 False，非空为 True

log("INFO", "用户登录", user_id=1001)
# level="INFO", msg="用户登录", args=(), kwargs={"user_id": 1001}
log("ERROR", "请求失败", 404, "Not Found", retry=3)
# level="ERROR", msg="请求失败", args=(404, "Not Found"), kwargs={"retry": 3}

---

## ⚠️ 惊天大坑：可变默认参数

这是 Python 面试中**被问得最多的陷阱**，没有之一。

### 现象

```python
def add_item(item, lst=[]):
    lst.append(item)
    return lst

print(add_item(1))  # [1]
print(add_item(2))  # [1, 2]  ← 出问题了！不应该是 [2] 吗？
```

### 为什么会出现这个现象？——深入解释器层面

**默认值在函数定义时只计算一次，然后被"冻结"在函数对象上。**

当你写下 `def add_item(item, lst=[]):` 时，Python 做了这么几件事：

1. **编译阶段**：Python 编译这段代码，识别出 `lst` 有一个默认值，这个默认值是一个列表字面量 `[]`
2. **函数定义执行阶段**：当 Python 执行到 `def` 语句时（注意，这发生在函数被调用之前！），它创建了一个函数对象，然后**计算默认参数值**——执行 `[]`，在堆内存上创建一个空列表对象
3. **绑定默认值**：Python 把这个列表对象的引用存入函数对象的 `__defaults__` 属性中。这个属性是一个元组，存储所有默认参数的值。`add_item.__defaults__` 会返回 `([],)`
4. **后续调用**：以后每次调用这个函数而不提供 `lst`，Python 从 `__defaults__` 中取出那个列表对象，赋值给 `lst`

关键点在于，那个列表对象从第二步开始就存在于内存中，没有被销毁。所有不传 `lst` 的调用，使用的都是**同一个列表对象**。每次 `.append()` 都在修改这个共享对象，而不是创建新列表。

### 为什么 Python 要这样设计？

你可能觉得这是个"Bug"，但实际上是设计选择：

- **性能**：如果每次调用都重新计算默认值，对于复杂的默认表达式（比如 `def f(x, cache=load_big_file())`），会造成巨大的性能浪费。默认值只在定义时求值一次，之后直接复用。
- **可预见的**：默认值在定义时求值，行为是可预测的——你可以在任何时间检查 `func.__defaults__` 来看当前默认值是什么。这让你能调试和检查函数的状态。
- **Python 之禅"显式优于隐式"**：Python 把这个机制暴露给你，而不是隐藏起来，让你清楚默认值的生命周期。

但副作用就是：可变默认值会被修改。这是 Python 设计中最著名的"陷阱"之一，也是面试官最爱问的问题。

### 验证

```python
print(add_item.__defaults__)  # 看看默认值是什么
# 如果你调用过两次，这里显示 ([1, 2],)
# 证明默认值确实被修改了
```

### 安全写法：用 None 代替可变默认值

```python
def add_item(item, lst=None):
    if lst is None:
        lst = []          # 每次调用创建新列表
    lst.append(item)
    return lst
```

**把 `None` 作为默认值**，在函数内部判断并创建新对象。因为 `None` 是不可变的，不会出现"共享修改"的问题。这个模式如此常见，以至于很多 Python 开发者看到 `def func(x=None)` 就能立刻意识到作者的意图：这个参数是可选的，默认值会在函数内部创建。

### 更多需要警惕的可变默认值

不仅列表有这个问题。任何可变类型作为默认值都会踩坑：
- `def f(d={})` — 字典作为默认值，多次调用会累积键值对
- `def f(s=set())` — 集合作为默认值
- `def f(obj=MyMutableClass())` — 自定义可变对象作为默认值

**黄金法则**：默认值只用不可变类型——`None`、`int`、`float`、`str`、`tuple`、`bool`。如果需要一个可变类型的默认值，用 `None` 并在函数内部创建。

In [ ]:
# ================================
# 可变默认参数陷阱——完整演示
# ================================
# 这是 Python 面试中最高频的陷阱问题
# 核心原因：默认参数值在函数定义时只计算一次，之后所有调用共享同一个可变对象

# 错误示范
def add_student(name, roster=[]):
    """把学生加入花名册（有 Bug 版本）"""
    # roster=[]：默认参数是一个空列表
    # 问题：这个列表在函数定义时创建一次，之后所有未提供 roster 的调用共享同一个列表
    # 可以通过 add_student.__defaults__ 查看函数默认值元组
    roster.append(name)
    # 列表是可变的，.append() 会原地修改列表对象
    # 由于所有未传 roster 的调用共享同一个列表对象，每次 append 都会永久改变它
    return roster

print("第1次调用:", add_student("张三"))  # ['张三']
# 第一次调用：roster 使用默认的空列表 []，append("张三") 后为 ['张三']
print("第2次调用:", add_student("李四"))  # ['张三', '李四'] ← 有李四的残留！
# 第二次调用：roster 仍然使用同一个默认列表（已经被改为 ['张三']）
# append("李四") 后变成 ['张三', '李四'] ← 这不是期望的结果！
print("第3次调用:", add_student("王五"))  # ['张三', '李四', '王五']
# 第三次调用继续追加，列表越来越长

# 查看默认值——已经被改了！
print("默认值被改成了:", add_student.__defaults__)
# __defaults__ 是函数对象的属性，存储所有默认参数的当前值
# 输出：(['张三', '李四', '王五'],) —— 默认值已经被调用过程修改了！
# 这说明默认值确实是同一个对象，并且被函数内部的 .append() 改变了

print()

# 正确示范
def add_student_safe(name, roster=None):
    """把学生加入花名册（安全版本）"""
    # roster=None：使用 None（不可变类型）作为默认值
    # None 是 NoneType 的唯一值，是不可变对象，不存在被修改的风险
    if roster is None:
        roster = []    # 每次新列表
    # 关键：每次调用时，如果没传 roster，都会创建一个全新的空列表
    # 这样每次调用都是独立的列表，互不干扰
    # 使用 is 判断而非 ==，因为 None 是单例，is 比 == 更快更安全
    roster.append(name)
    return roster

print("第1次safe:", add_student_safe("张三"))  # ['张三']
print("第2次safe:", add_student_safe("李四"))  # ['李四'] ← 正确！独立的列表
# 每次调用都创建新列表，互不影响

# 但显式传列表时，我们确实想往里面加
group = ["赵六"]
print("传入已有列表:", add_student_safe("钱七", group))  # ['赵六', '钱七']
# 显式传入 roster=group，不经过 None 判断分支
# 此时 roster 指向 group 指向的同一个列表对象
print("原列表被修改:", group)  # ['赵六', '钱七']
# 因为 roster 和 group 是同一个对象的引用，所以原列表也被修改了
# 这是预期的行为：显式传入列表时，我们希望它被原地修改

# ⚠️ 同理的陷阱：不要用 {} 做默认值，不要用 set() 做默认值
# 规则很简单：默认值只用不可变类型（None, int, str, tuple, bool）
# 可变类型（list, dict, set）做默认值都会存在这个共享修改问题

---

## 作用域与 LEGB 规则

### 什么是作用域（Scope）？

作用域 = **变量在代码中"可见"的范围**。

想象一栋办公楼：
- **你的工位抽屉**：只有你能打开（局部变量）
- **部门的共享柜**：部门所有人都能用（闭包变量）
- **公司的公告栏**：全公司都看得到（全局变量）
- **城市的路牌**：所有人默认使用（内置函数/类型）

Python 按 **LEGB** 顺序查找变量，找到第一个就停止。

### LEGB 详解

| 层级 | 名称 | 说明 | 包含 |
|------|------|------|------|
| **L** | Local | 当前函数内部 | 函数参数、函数内赋值的变量 |
| **E** | Enclosing | 外层函数 | 嵌套函数中外层函数的变量 |
| **G** | Global | 模块顶层 | 模块级别定义的变量 |
| **B** | Built-in | Python 内置 | `print`、`len`、`int` 等 |

### Python 为什么这样设计作用域？

LEGB 规则不是随意制定的——它解决了几个核心问题：

1. **封装性**：局部变量在函数外部不可见，函数是一个"黑盒"。你调用 `len()` 时不需要知道它内部用了什么临时变量——这些变量是 `len` 的内部实现细节，不会泄漏到外部。

2. **模块隔离**：每个模块都有自己的全局作用域。你在 `a.py` 中定义的 `x = 1` 和 `b.py` 中的 `x = 2` 不会冲突，因为它们是不同模块的全局变量。

3. **安全性**：函数默认不能修改全局变量，这防止了意外的状态污染。你必须显式使用 `global` 声明，这让你清楚地意识到自己在修改全局状态。

### 为什么不能直接修改外层变量？

当你写 `x = 5` 时，Python 认为是**在当前作用域定义一个新变量**。如果你想**修改**外层变量，需要用 `global` 或 `nonlocal` 声明。这个设计是为了防止函数意外修改外部状态——函数应该是"安全的"，不能随便改变全局变量。

### 常见错误 1：遮蔽内置函数

```python
len = 10        # 覆盖了内置函数 len
print(len([1, 2, 3]))  # TypeError: 'int' object is not callable
```

当你写了 `len = 10`，Python 在全局作用域中创建了一个变量 `len`，遮蔽了内置作用域中的 `len` 函数。从此，这个模块中所有对 `len` 的引用都指向整数 10 而不是内置函数。这不仅影响到你自己写的代码，还可能影响到你调用的其他函数——如果某个函数内部没有自己的局部 `len`，它也会找到你定义的全局 `len`，然后崩溃。

**不要用 `list`、`str`、`len`、`sum`、`type`、`dict`、`set`、`tuple`、`int`、`float`、`bool`、`input`、`print`、`open`、`file` 等内置名称做变量名。**

### 常见错误 2：闭包中的变量捕获陷阱

```python
funcs = []
for i in range(3):
    funcs.append(lambda: i)

for f in funcs:
    print(f())  # 输出 2, 2, 2，不是 0, 1, 2！
```

这是因为 lambda 捕获的是变量 `i` 的引用，而不是创建时的值。当循环结束时，`i` 的值是 2，三个 lambda 都引用同一个 `i`。这个问题可以通过默认参数解决：`lambda i=i: i`。

### 常见错误 3：在条件语句中意外创建全局变量

```python
def process():
    print(value)  # 可以读取全局变量
    value = 10   # 下一行就报错！Python 认为 value 是局部变量
```

Python 编译时扫描函数体，发现 `value` 被赋值，就把它标记为局部变量。但 `print(value)` 执行时，局部 `value` 还没被赋值，所以抛出 `UnboundLocalError`。这再次说明：**读取全局变量可以，但不要试图在函数中混用同名的局部变量和全局变量。**

In [ ]:
# ================================
# LEGB 规则逐层演示
# ================================
# LEGB：Local（局部）→ Enclosing（闭包）→ Global（全局）→ Built-in（内置）
# Python 在查找变量时按此顺序搜索，找到第一个就停止

x = "全局 x"  # G — 全局
# 这个 x 定义在模块最顶层，属于全局作用域
# 整个模块（包括所有函数内部，除非被局部变量遮蔽）都可以访问

def outer():
    x = "外层 x"  # E — 闭包
    # 这个 x 定义在 outer 函数内部，属于闭包作用域
    # 它遮蔽了全局的 x（全局的 x 在 outer 函数内部不可见了）
    # 对于 inner 函数来说，outer 函数的变量就是它的 Enclosing 层

    def inner():
        x = "内层 x"  # L — 局部
        # 这个 x 定义在 inner 函数内部，属于局部作用域
        # 它同时遮蔽了外层的 x 和全局的 x
        # 局部作用域只在当前函数执行期间存在，函数结束后销毁
        print("inner 看到:", x)  # 内层 x
        # LEGB 查找：先在 inner 局部找到 x → "内层 x"，立即返回，不再往上找

    inner()
    print("outer 看到:", x)      # 外层 x
    # inner 内的 x 不影响 outer 的 x（它们是不同的作用域）
    # outer 在自己作用域内查找 x → 找到 "外层 x"

outer()
print("全局 看到:", x)          # 全局 x
# 全局作用域中的 x 没有被任何函数影响，仍然是 "全局 x"

# 如果去掉 inner 的 x，inner 会找到 outer 的 x（E 层）
# 如果再去掉 outer 的 x，会找到全局 x（G 层）
# 如果全局也没有，找内置（B 层）

# 遮蔽演示：别用内置名字做变量！
# len = "我不应该存在"  # 取消注释这行试试
# 这行将 len 这个名字绑定到了字符串上，遮蔽了内置函数 len()
# print(len([1, 2, 3]))  # TypeError！
# Python 在 LEGB 中查找 len：
#   1. 局部没有 → 2. 闭包没有 → 3. 全局找到 "我不应该存在"（字符串）
#   4. 对字符串执行 ([1,2,3]) 调用 → TypeError: 'str' object is not callable
# 所以：不要用 list, str, dict, len, sum, type 做变量名
# 一旦遮蔽了内置函数，恢复需要 del 变量或重启解释器

In [ ]:
# ================================
# global 关键字
# ================================
# global 用于在函数内部声明：我要修改的是全局变量，而不是创建新的局部变量
# 如果不加 global，Python 认为函数内的赋值语句是定义一个新局部变量

count = 0  # 全局变量
# 定义在模块最顶层，属于全局作用域
# 默认情况下，函数内可以"读取"全局变量，但不能"修改"（赋值）

def increment():
    global count  # 声明：我要修改全局变量
    # 告诉 Python：在这个函数中，count 指的是全局的那个 count
    # 没有这行的话，count += 1 会引发 UnboundLocalError
    # 原因：Python 在编译时发现函数内有对 count 的赋值（count += 1）
    # 就认为 count 是局部变量，但执行 count += 1 时 count 还没有定义
    count += 1
    # count += 1 等价于 count = count + 1
    # 如果没有 global 声明，右边的 count 被认为是未初始化的局部变量

print("初始:", count)
increment()
# 调用 increment()，由于 global 声明，全局 count 从 0 变为 1
print("调用后:", count)
# 输出 1，全局变量被成功修改

# 如果不加 global：
def broken_increment():
    count = count + 1  # UnboundLocalError！
    # Python 认为 count 是局部变量，但还没定义就使用了
    # 原因：函数内部有 count = ... 赋值操作
    # Python 编译器将 count 标记为局部变量
    # 但赋值右侧的 count 还没来得及绑定值
    # 因此抛出 UnboundLocalError: local variable 'count' referenced before assignment

# ⚠️ 尽量少用 global！
# 全局变量在多处修改 → 代码难以理解 → Bug 难以追踪
# 建议：参数传递 + 返回值 通常更好的选择
# 使用 global 的场景：计数器、缓存、配置标志等少数情况
# 更优雅的替代方案：将需要共享的状态封装在类或闭包中

---

## lambda：匿名函数

lambda 创建**没有名字、一行写完的小函数**。

### 语法

```python
lambda 参数: 表达式
```

等价于：

```python
def 函数名(参数):
    return 表达式
```

### 为什么叫"匿名函数"？

lambda 表达式的结果是一个函数对象，但它没有 `__name__`（或者说 `__name__` 是 `"<lambda>"`）。它就是"用完即弃"的临时工具。

### lambda 与 def 的本质区别

了解 lambda 和 def 在"里子"上的区别，能帮你做出更好的选择：

| 对比项 | `def` | `lambda` |
|--------|-------|----------|
| **名称** | 有 `__name__` | `__name__` 为 `"<lambda>"` |
| **内容** | 可以包含任意多条语句 | 只能包含单个表达式 |
| **返回值** | 用 `return` 显式返回 | 自动返回表达式结果 |
| **复用** | 设计上就是为复用 | 设计上是一次性的 |
| **文档** | 可以写 docstring | 没法写文档字符串 |
| **适用** | 任何复杂逻辑 | 一眼能看懂的简单变换 |

最关键的区别在于：**lambda 只能包含一个表达式，不能包含语句**。这意味着你不能在 lambda 中使用赋值（`=`）、循环（`for`）、条件分支（`if/elif/else`，但三元表达式 `x if cond else y` 可以）或 `return`。任何需要多行逻辑的场景，都应该用 `def`。

### 什么时候用 lambda？

1. **传给 `sorted`/`filter`/`map` 作为 key 或判断条件**
2. **临时需要一个简单函数，不值得用 `def`**
3. **GUI 回调、事件处理**

### 什么时候不要用 lambda？

1. **逻辑超过一个表达式** — 不要试图在 lambda 里写复杂逻辑。如果你需要多行代码，用 `def`。可读性比代码简短更重要。
2. **你需要复用这段逻辑** — lambda 设计上就是一次性的。如果你要调用同一个 lambda 两次以上，用 `def` 定义成命名函数更清晰。
3. **你需要文档字符串** — lambda 没有地方写 docstring。
4. **调试时需要清晰的堆栈追踪** — traceback 中 `def` 函数会显示函数名，而 lambda 只显示 `<lambda>`，不利于定位问题。

一句话：**如果你需要用变量名保存 lambda 才能让代码可读，那它就应该是一个 `def`。**

In [ ]:
# ================================
# lambda 实战
# ================================
# lambda 关键字创建匿名函数：lambda 参数: 表达式
# 特点：没有函数名（__name__ 为 "<lambda>"）、只能包含单个表达式（不能有语句）
# 适用场景：作为高阶函数的参数（如 sorted 的 key、filter/map 的回调）

# 基本语法
square = lambda x: x ** 2
# lambda x: x ** 2：定义一个匿名函数，接收参数 x，返回 x ** 2
# 这里将匿名函数赋值给变量 square，相当于 def square(x): return x ** 2
print("lambda:", square(5))   # 25
# 通过变量 square 调用 lambda 函数

# 等价于：
def square_def(x):
    return x ** 2
print("def:  ", square_def(5))
# 传统 def 语句定义的函数，效果完全相同

print()

# --- sorted + lambda：按指定规则排序 ---
students = [
    ("张三", 85),
    ("李四", 92),
    ("王五", 78),
    ("赵六", 95),
]
# 这是一个列表，每个元素是一个包含姓名和分数的元组

# 默认按名字（元组第一个元素）排序
# 按分数排序
by_score = sorted(students, key=lambda s: s[1], reverse=True)
# sorted(iterable, key, reverse)：返回一个新的排序后的列表
# key=lambda s: s[1]：传入一个函数作为排序依据
#   s 是列表中的每个元素（元组），s[1] 是元组的第二个元素（分数）
#   sorted 对每个元素调用 key 函数，用返回值比较大小
# reverse=True：降序排列（分数从高到低）
print("按分数从高到低:")
for name, score in by_score:
    print(f"  {name}: {score}")

print()

# --- filter + lambda：筛选 ---
nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
evens = list(filter(lambda x: x % 2 == 0, nums))
# filter(function, iterable)：对 iterable 中每个元素应用 function
# 保留 function 返回 True 的元素
# lambda x: x % 2 == 0：判断 x 是否为偶数
# filter 返回一个迭代器，需要用 list() 转换为列表
print("偶数:", evens)

# --- map + lambda：变换 ---
squares = list(map(lambda x: x ** 2, nums))
# map(function, iterable)：对 iterable 中每个元素应用 function
# 返回一个新的迭代器，包含所有变换后的结果
# lambda x: x ** 2：计算 x 的平方
# 同样需要用 list() 转换为列表
print("平方:", squares)

# 等价列表推导（通常更清晰！）
print("列表推导:", [x ** 2 for x in nums])
# 列表推导式 [x ** 2 for x in nums] 比 map + lambda 更 Pythonic
# 规则：能用列表推导解决的问题，优先使用列表推导
# lambda 用得过多会降低代码可读性

---

## 模块：把代码组织成文件

### 什么是模块？

**模块 = `.py` 文件**。当你写了一个 `utils.py`，它就变成了 `utils` 模块，其他 Python 文件可以 `import` 它。

### 为什么需要模块？

1. **组织代码** — 把相关功能放在一个文件里（数学函数放 `math_utils.py`）
2. **命名空间** — 不同模块的同名函数不会冲突（`math.sqrt` vs `numpy.sqrt`）
3. **复用** — 写一次，到处用

### 四种 import 写法

| 写法 | 调用方式 | 推荐度 |
|------|----------|--------|
| `import 模块` | `模块.函数()` | 最推荐 |
| `import 模块 as 别名` | `别名.函数()` | 模块名太长时 |
| `from 模块 import 函数` | 直接 `函数()` | 谨慎使用 |
| `from 模块 import *` | 直接所有 | 不推荐 |

前两种保留了命名空间，最安全。

### sys.path：Python 去哪里找模块？

当你写 `import math` 时，Python 在 `sys.path` 列出的路径中搜索 `math.py`：

```python
import sys
print(sys.path)
# 输出类似：
# ['当前目录', '/usr/lib/python3.x', '/usr/lib/python3.x/site-packages', ...]
```

搜索顺序是：**当前目录 → 标准库路径 → site-packages（第三方库）**。这就是为什么你的脚本文件名不能叫 `random.py`——如果你创建了一个 `random.py`，Python 会在当前目录先找到它，而不是标准库的 `random` 模块。这叫做"模块遮蔽"，是一个常见的初学者陷阱。

### 命名空间污染：from ... import * 为什么危险？

当你写 `from math import *` 时，所有不以下划线开头的 math 模块中的名字都被导入到当前命名空间。这意味着：

1. **你不知道导入了什么** — 代码中突然多了几十个你不一定知道的函数名
2. **覆盖风险** — 如果你恰好定义了一个同名的变量或函数，会静默覆盖导入的函数
3. **后续代码难以理解** — 看到 `sqrt(4)`，读者不知道 sqrt 是从哪里来的

这也是为什么 PEP 8 建议：`import *` 只在交互式环境或特定场景（如 `from tkinter import *`）中使用。在日常开发中，永远用 `import 模块` 或 `from 模块 import 特定名称`。

### 循环导入：你导入了我，我导入了你

假设有两个文件：

```python
# a.py
from b import func_b
def func_a():
    func_b()

# b.py
from a import func_a
def func_b():
    func_a()
```

当你运行 `a.py` 时，Python 执行 `from b import func_b`，开始加载 `b.py`。`b.py` 的第一行是 `from a import func_a`，但此时 `a.py` 还没执行完——`func_a` 还没有被定义！于是循环导入报错：`ImportError: cannot import name 'func_a' from 'a'`。

解决方案：
- **重构代码**，把共享的依赖提取到第三个模块中（最推荐）
- **延迟导入**，在函数内部 `import` 而不是在模块顶层
- **使用 `import 模块` 而非 `from 模块 import 名字`**，因为模块级导入在模块加载完成后才尝试属性访问

### Python 标准库：内置电池

Python 自带大量模块，不需要额外安装。最常用的几个：
- **`math`**：数学函数和常量
- **`random`**：随机数生成
- **`datetime`**：日期时间处理
- **`json`**：JSON 数据编解码
- **`os`**：操作系统接口
- **`sys`**：Python 解释器接口

In [ ]:
# ================================
# import 的四种写法
# ================================
# 模块导入是将其他 .py 文件中的代码引入当前命名空间的机制
# 每种导入方式有不同的命名空间影响，选择合适的写法很重要

# 写法 1：完整导入（最推荐）
import math
# import 模块名：将整个模块导入，通过 模块名.函数 的方式调用
# 优点：保留了命名空间，不会与其他模块的同名函数冲突
# 例如 math.sqrt 和 numpy.sqrt 可以共存
print(f"√2 ≈ {math.sqrt(2):.4f}")
# math.sqrt(2)：调用 math 模块中的 sqrt 函数计算平方根
# :.4f 格式化：保留 4 位小数
print(f"π  ≈ {math.pi:.4f}")
# math.pi：模块常量，圆周率 π

# 写法 2：起别名
import datetime as dt
# import 模块名 as 别名：导入模块并赋予简短别名
# 适用：模块名较长或需要避免命名冲突时
# datetime 是标准库模块，as dt 让后续调用更简洁
today = dt.date.today()
# dt.date.today()：通过别名调用，获取当前日期（不包含时间）
print(f"今天: {today}")

# 写法 3：只导入需要的（省打字，但可能冲突）
from random import randint, choice
# from 模块 import 名称：只导入模块中的特定函数/变量
# 这样调用时直接写函数名，不需要 模块名. 前缀
# 风险：如果当前模块有同名变量，会覆盖导入的函数
print(f"掷骰子: {randint(1, 6)}")
# randint(1, 6)：返回 1 到 6（包含两端）之间的随机整数
print(f"随机选: {choice(['石头', '剪刀', '布'])}")
# choice(seq)：从序列中随机选择一个元素

# 写法 4：from ... import *（不推荐！）
# from math import *
# * 表示导入模块中所有公开的名称（不以下划线开头的）
# 问题：大量名称被注入当前命名空间，极易造成命名冲突
# 如果当前模块有同名的变量或函数，会被静默覆盖！
# sqrt = "我不小心覆盖了 math.sqrt"  # 悲剧了
# print(sqrt(4))  # TypeError
# 上面两行被注释掉了，但可以想象：
# 如果先 from math import * 导入了 sqrt
# 然后定义 sqrt = "字符串"，就覆盖了 math.sqrt
# 后续调用 sqrt(4) 会报错，因为字符串不可调用
# 最佳实践：永远不要在生产代码中使用 from ... import *

In [ ]:
# ================================
# 标准库精选演示
# ================================
# Python 标准库提供丰富的内置模块，无需额外安装即可使用
# 本段演示最常用的四个标准库：math、random、datetime、json

# --- math ---
import math
# math 模块提供数学运算函数和常量
print("=== math 模块 ===")
print(f"  π = {math.pi:.6f}")
# math.pi：圆周率常量 π = 3.141592...
print(f"  e = {math.e:.6f}")
# math.e：自然常数 e = 2.718281...
print(f"  sin(30°) = {math.sin(math.radians(30)):.2f}")
# math.radians(30)：将角度 30° 转换为弧度（π/6 ≈ 0.5236）
# math.sin(...)：计算正弦值，sin(30°) = 0.5
# 嵌套调用：内层 math.radians() 先执行，结果传入外层 math.sin()
print(f"  floor(3.7) = {math.floor(3.7)}")
# math.floor(3.7)：向下取整 → 3（向负无穷方向取整）
print(f"  ceil(3.2) = {math.ceil(3.2)}")
# math.ceil(3.2)：向上取整 → 4（向正无穷方向取整）

print()

# --- random ---
import random
# random 模块提供各种随机数生成函数
print("=== random 模块 ===")
print(f"  0~1 随机小数: {random.random():.3f}")
# random.random()：返回 [0.0, 1.0) 范围内的随机浮点数（含 0 不含 1）
print(f"  1~100: {random.randint(1, 100)}")
# random.randint(a, b)：返回 [a, b] 范围内的随机整数（两端包含）
cards = ["A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K"]
random.shuffle(cards)  # 洗牌
# random.shuffle(list)：原地打乱列表顺序（直接修改原列表，不返回新列表）
# 注意：shuffle 没有返回值（返回 None），所以不能写 cards = random.shuffle(cards)
print(f"  洗牌: {cards}")
print(f"  抽两张: {random.sample(cards, 2)}")
# random.sample(population, k)：从序列中随机抽取 k 个不重复的元素
# 返回一个新的列表，原列表不变

print()

# --- datetime ---
import datetime as dt
# datetime 模块提供日期和时间的处理功能
print("=== datetime 模块 ===")
now = dt.datetime.now()
# dt.datetime.now()：获取当前本地日期时间（包含年、月、日、时、分、秒、微秒）
print(f"  现在: {now}")
print(f"  年: {now.year}, 月: {now.month}, 日: {now.day}")
# 通过 .year / .month / .day 等属性访问日期时间的各个分量
one_week = dt.timedelta(days=7)
# dt.timedelta(days=7)：创建一个时间差对象，表示 7 天
# timedelta 参数支持 days、hours、minutes、seconds、microseconds 等
print(f"  一周后: {now + one_week}")
# datetime + timedelta：日期时间运算，得到新的日期时间
# 字符串格式化
print(f"  格式化: {now.strftime('%Y年%m月%d日 %H:%M:%S')}")
# strftime(format)：将日期时间格式化为指定格式的字符串
# %Y = 四位年份，%m = 两位月份，%d = 两位日期
# %H = 24小时制小时，%M = 分钟，%S = 秒
# 注意：中文"年月日"字符直接嵌入格式字符串中

print()

# --- json ---
import json
# json 模块用于 JSON 数据的编码（Python → JSON）和解码（JSON → Python）
print("=== json 模块 ===")
person = {
    "name": "小明",
    "age": 25,
    "skills": ["Python", "数据分析", "机器学习"],
    "active": True
}
# Python 字典：包含字符串、整数、列表、布尔值等多种类型
json_str = json.dumps(person, ensure_ascii=False, indent=2)
# json.dumps(obj, ensure_ascii, indent)：将 Python 对象序列化为 JSON 字符串
# ensure_ascii=False：允许输出非 ASCII 字符（中文正常显示）
#   默认为 True，中文会被转义为 \uXXXX 形式
# indent=2：使用 2 个空格缩进格式化输出，便于阅读
#   indent=None（默认）输出紧凑格式，无缩进和换行
print("JSON 输出:")
print(json_str)

# 反序列化：字符串 → Python 对象
recovered = json.loads(json_str)
# json.loads(json_string)：将 JSON 字符串解析为 Python 对象
# JSON 到 Python 的类型映射：object → dict, array → list, string → str
#   number → int/float, boolean → bool, null → None
print(f"名字: {recovered['name']}, 技能数: {len(recovered['skills'])}")
# 反序列化后的 recovered 是一个字典，可以正常通过 key 访问

---

## `if __name__ == "__main__"` 模式

这是 Python 中最常见、也最被误解的惯用模式之一。

### 核心问题

每个 `.py` 文件在被 `import` 时，Python **会从头到尾执行整个文件**。如果你的文件里有测试代码、演示代码，别人导入时这些代码也会执行——这通常不是你想要的结果。

### 解决方案

```python
def main():
    # 主程序逻辑
    pass

if __name__ == "__main__":
    main()
```

### 原理

Python 在执行每个文件时，会给文件设置一个名为 `__name__` 的变量：

| 执行方式 | `__name__` 的值 |
|----------|----------------|
| 直接运行文件 | `"__main__"` |
| 被其他文件 `import` | 文件名（不含 `.py`） |

所以 `if __name__ == "__main__"` 就是在问：**我是被直接运行的，还是被导入的？**

### 模块缓存：import 只执行一次

你可能担心：如果一个模块被多处 `import`，会不会反复执行多次？答案是不会。Python 有模块缓存机制：首次 `import` 一个模块时，Python 会执行该模块的代码，并将模块对象缓存到 `sys.modules` 字典中。之后对同一模块的 `import`，直接返回缓存，不再重复执行。

```python
import sys
print('math' in sys.modules)  # False，第一次
import math                    # 执行 math 模块代码
print('math' in sys.modules)  # True，已缓存
import math                    # 直接从缓存取，不执行代码
```

这意味着即使你在模块顶层写了循环导入，Python 的模块缓存机制也能避免无限递归——但缓存中的模块可能不完整（还在加载中），这反而可能引发更隐蔽的问题。

### 三个好处

1. **双用途模块** — 一个文件既可以当脚本直接运行，也可以被导入复用
2. **测试友好** — 每个模块都能独立测试
3. **干净导入** — 导入时不执行测试/演示代码

In [ ]:
# ================================
# __name__ 深度演示
# ================================
# __name__ 是每个 Python 模块自动拥有的内置变量
# 当文件直接运行时 __name__ 被设为 "__main__"
# 当文件被 import 时 __name__ 被设为模块名（即文件名不含 .py）

print(f"当前 __name__ 的值: {__name__!r}")
# __name__!r：!r 是 repr() 转换标志，输出带引号的字符串表示
# 在 Jupyter 中运行单元格时 __name__ 通常是 "__main__"
# 注意：Jupyter 内部机制特殊，__name__ 值取决于执行上下文

def main():
    """程序入口"""
    print("这是主程序！")
# 将主要逻辑封装在 main() 函数中是一种良好实践
# 这样当模块被导入时，这些代码不会自动执行

# 只有直接运行时才执行 main()
if __name__ == "__main__":
    # 条件判断：检查当前模块是直接运行还是被导入
    # "__main__" 是 Python 为直接运行的模块设置的特殊值
    # 注意：这里比较的是字符串 "__main__"，不是变量 __main__
    print("→ 直接运行模式")
    main()
else:
    # 当本文件被其他文件 import 时，走这个分支
    print("→ 被导入模式")
    print("  模块函数已就绪")
    # 在这种模式下，main() 不会自动执行
    # 但模块中的函数和变量可以被导入方使用

# 每个 Python 文件都自动有 __name__ 变量
# 试试在终端运行 python 04_functions.ipynb（通过 jupyter）
# 然后 from 04_functions import something
# 你会发现 __name__ 不同！
# 这个模式的价值：
#   1. 模块即可作为独立脚本运行，也可被导入复用
#   2. 测试代码放在 if __name__ == "__main__" 块中
#   3. 导入时不会运行测试/演示代码，保持干净

---

## 🎯 本课总结

| 概念 | 核心要点 |
|------|----------|
| **DRY 原则** | 不要重复自己，函数是复用的基本单位 |
| **函数本质** | 调用时创建帧对象、压入调用栈、参数绑定后执行字节码、返回时销毁帧对象 |
| **函数定义** | `def` 关键字，参数列表，函数体，`return` 返回值 |
| **return** | 没有 return = 返回 `None`，return 后代码不执行，用 `is None` 判断 |
| **参数** | 位置 → 默认 → 关键字 → `*args` → `**kwargs`（按此顺序） |
| **参数设计哲学** | 每种参数类型解决特定问题，体现"显式优于隐式" |
| **可变默认值陷阱** | 默认值在函数定义时只计算一次并冻结在 `__defaults__` 中，用 `None` + 内部判断替代 |
| **LEGB 规则** | Local → Enclosing → Global → Built-in，按此顺序查找变量，遮蔽是常见错误 |
| **lambda** | 匿名函数，只能包含单个表达式，适合简短的一次性逻辑，复杂逻辑用 `def` |
| **模块** | `.py` 文件就是模块，用 `import` 导入，`sys.path` 决定搜索顺序 |
| **`__name__`** | 区分直接运行和导入，实现模块两用 |
| **安全红线** | 不要用内置名称做变量名；默认值只用不可变类型；避免循环导入 |

---

## 🧪 最终练习：密码生成器

把今天学的内容串起来，写一个密码生成器。

**要求：**
- 函数 `generate_password(length=12, use_digits=True, use_special=True)`
- 默认长度 12，可选包含数字和特殊字符
- 使用 `random` 和 `string` 标准库模块
- 使用 `if __name__ == "__main__"` 模式

`string` 模块提供：
- `string.ascii_letters` → 大小写字母
- `string.digits` → `0123456789`
- `string.punctuation` → `!@#$%^&*` 等符号

In [ ]:
# ================================
# 最终练习：密码生成器
# ================================
# 综合运用本课所学：函数定义、默认参数、标准库导入、__name__ 模式

import random
import string
# random 模块：提供随机选择、打乱等函数
# string 模块：提供常用字符常量
#   string.ascii_letters：大小写字母 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
#   string.digits：数字 '0123456789'
#   string.punctuation：标点符号 '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

def generate_password(length=12, use_digits=True, use_special=True):
    """生成随机密码

    参数:
        length: 密码长度（默认12）
        use_digits: 是否包含数字
        use_special: 是否包含特殊字符

    返回:
        生成的密码字符串
    """
    # 参数说明：
    #   length：使用默认参数，大多数情况下 12 位密码足够安全
    #   use_digits 和 use_special：布尔类型默认参数，控制密码字符集组成
    # 返回类型：str

    # 从字母开始
    chars = string.ascii_letters  # 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
    # 初始字符集包含所有大小写字母，共 52 个字符

    if use_digits:
        chars += string.digits    # '0123456789'
    # 如果 use_digits 为 True，将数字追加到字符集中
    # += 运算符对字符串执行拼接，创建新的字符串对象
    # 条件选择使密码生成策略灵活可配

    if use_special:
        chars += string.punctuation  # '!"#$%...' 等
    # 如果 use_special 为 True，将标点符号追加到字符集中
    # 包含特殊字符的密码更难以被暴力破解

    # 从字符集中随机选择 length 个字符
    password = ''.join(random.choice(chars) for _ in range(length))
    # random.choice(chars)：从 chars 字符串中随机选择一个字符
    # for _ in range(length)：重复 length 次，_ 是忽略循环变量的惯用写法
    # 生成器表达式生成 length 个随机字符
    # ''.join(...)：用空字符串将所有随机字符连接成一个字符串
    # 执行流程：
    #   1. range(length) 生成 0 到 length-1 的整数序列
    #   2. 每次循环调用 random.choice(chars) 获取一个随机字符
    #   3. 所有字符传给 ''.join() 拼接为最终密码
    return password

def main():
    """演示密码生成器"""
    # main() 函数封装了演示逻辑，不会在导入时自动执行
    print("=" * 40)
    print("       密码生成器")
    print("=" * 40)
    # 使用字符串乘法生成分隔线，= * 40 生成 40 个等号

    print(f"\n默认12位:  {generate_password()}")
    # 全部使用默认参数：length=12, use_digits=True, use_special=True
    print(f"仅字母8位:  {generate_password(8, False, False)}")
    # 传入 length=8, use_digits=False, use_special=False → 纯字母密码
    print(f"含数字16位: {generate_password(16, True, False)}")
    # 传入 length=16, use_digits=True, use_special=False → 字母+数字
    print(f"全功能20位: {generate_password(20, True, True)}")
    # 所有选项全开，20 位高强度密码

    # 批量生成多个密码
    print("\n--- 批量生成 ---")
    for i in range(5):
        print(f"  密码{i+1}: {generate_password()}")
    # 循环调用 generate_password() 生成 5 个不同的随机密码
    # 由于 random 模块的随机性，每次调用结果都不同

if __name__ == "__main__":
    main()
# if __name__ == "__main__" 模式：
#   直接运行本文件时，__name__ 等于 "__main__"，条件为真，执行 main()
#   当本文件被其他模块 import 时，__name__ 等于 "04_functions"
#   条件为假，main() 不会自动执行，但 generate_password() 函数可以被导入使用